In [8]:
import time

import torch
from transformers import AutoModelForCausalLM

from neuralzip import Compressor

In [9]:
model = AutoModelForCausalLM.from_pretrained(
    "openai-community/gpt2",
    dtype=torch.bfloat16,
)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

## Compressors

In [10]:
neuralzip = Compressor(
    "neuralzip",
    keep_payloads=True
)

In [11]:
zipnn = Compressor(
    "zipnn",
    keep_payloads=True,
)

## Setup building

In [12]:
# Obtaining setups for reuse
setup = neuralzip.fit(model)
print(f"NeuralZip setup time: {setup.setup_report.setup_seconds:.4f} s")

NeuralZip setup time: 4.4537 s


## Encoding

In [15]:
# Compressing the model with neuralzip
compressed_model_neuralzip = neuralzip.encode(model, setup=setup)
encode_seconds_neuralzip = compressed_model_neuralzip.encode_seconds
compression_ratio_neuralzip = compressed_model_neuralzip.compression_ratio
print(
    f"NeuralZip — encode time: {encode_seconds_neuralzip:.4f} s | "
    f"compression ratio: {100 * compression_ratio_neuralzip:.2f}%"
)

NeuralZip — encode time: 0.0446 s | compression ratio: 67.06%


In [16]:
# Compressing the model with zipnn
compressed_model_zipnn = zipnn.encode(model)
encode_seconds_zipnn = compressed_model_zipnn.encode_seconds
compression_ratio_zipnn = compressed_model_zipnn.compression_ratio
print(
    f"ZipNN — encode time: {encode_seconds_zipnn:.4f} s | "
    f"compression ratio: {100 * compression_ratio_zipnn:.2f}%"
)

ZipNN — encode time: 0.0900 s | compression ratio: 66.59%


In [22]:
print(
    f"Speedup: {encode_seconds_zipnn / encode_seconds_neuralzip:.2f}x | "
    f"Compression ratio improvement: {compression_ratio_zipnn / compression_ratio_neuralzip:.2f}x"
)

Speedup: 2.02x | Compression ratio improvement: 0.99x


## GPU Inference

## Decoding

In [23]:
# Decompressing the model with neuralzip
decode_started = time.perf_counter()
decompressed_model_neuralzip = neuralzip.decode(compressed_model_neuralzip, setup=setup)
neuralzip_decode_seconds = time.perf_counter() - decode_started
print(f"NeuralZip decode time: {neuralzip_decode_seconds:.4f} s")

# Check if we recover the original model
for p1, p2 in zip(model.parameters(), decompressed_model_neuralzip.parameters()):
    assert torch.allclose(p1, p2, atol=1e-3), "Parameters are not equal!"

TypeError: Compressor.decode() got an unexpected keyword argument 'setup'

In [20]:
# Decompressing the model with zipnn
decode_started = time.perf_counter()
decompressed_model_zipnn = zipnn.decode(compressed_model_zipnn)
zipnn_decode_seconds = time.perf_counter() - decode_started
print(f"ZipNN decode time: {zipnn_decode_seconds:.4f} s")

# Check if we recover the original model
for p1, p2 in zip(model.parameters(), decompressed_model_zipnn.parameters()):
    assert torch.allclose(p1, p2, atol=1e-3), "Parameters are not equal!"

ZipNN decode time: 1.9837 s
